In [1]:
from json import dumps
import os

import datasets
from transformers import AutoTokenizer

from examples import find_texts, DATA_DIR

## Prototype

In [2]:
kwargs = {
    "neuron_subset_name": "weakening",
    "intervention_type": "mean_ablation",
    "topk": 4,
}

In [3]:
data_dir = os.path.join(
            DATA_DIR,
            kwargs["neuron_subset_name"],
            f"{kwargs["intervention_type"]}_None",
        )

In [5]:
#text_dataset = datasets.load_from_disk('neuroscope/datasets/dolma-small')
text_dataset = datasets.load_dataset('sjgerstner/dolma-small')

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/126M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45736 [00:00<?, ? examples/s]

In [10]:
text_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'id', 'metadata', 'input_ids', 'attention_mask'],
        num_rows: 45736
    })
})

In [11]:
text_dataset = text_dataset['train']

In [ ]:
indices = find_texts(
    metric="loss",
    data_dir=DATA_DIR,
    extremum="min",
    **kwargs,
)

In [12]:
my_input_ids = text_dataset['input_ids'][indices[:,0]]

In [17]:
my_input_ids

[[50279,
  22653,
  261,
  4137,
  46793,
  457,
  84,
  5284,
  5249,
  588,
  1705,
  281,
  271,
  990,
  846,
  10922,
  271,
  4345,
  342,
  521,
  5006,
  3639,
  5453,
  40322,
  15,
  380,
  4760,
  5262,
  1458,
  1107,
  275,
  3060,
  13,
  4882,
  323,
  443,
  1406,
  25203,
  13,
  47867,
  13,
  22808,
  13,
  24639,
  13,
  285,
  2058,
  1059,
  4173,
  21924,
  15,
  187,
  4553,
  521,
  767,
  14,
  13263,
  3310,
  342,
  2058,
  1059,
  4173,
  21924,
  275,
  11268,
  27416,
  13,
  344,
  3395,
  247,
  1959,
  5570,
  285,
  369,
  7296,
  14167,
  247,
  1180,
  273,
  6671,
  15,
  831,
  3977,
  281,
  247,
  24147,
  275,
  5841,
  432,
  40322,
  14501,
  26412,
  342,
  779,
  281,
  1091,
  281,
  521,
  806,
  2285,
  15,
  187,
  22653,
  261,
  4137,
  46793,
  556,
  644,
  2190,
  253,
  1755,
  3521,
  23521,
  275,
  253,
  1533,
  1580,
  4886,
  281,
  3060,
  275,
  5215,
  347,
  247,
  1384,
  14,
  2913,
  14,
  744,
  28713,
  13,
  14631,

In [22]:
print([len(x) for x in my_input_ids])

[172, 424, 84, 209]


In [13]:
tokenizer = AutoTokenizer.from_pretrained('allenai/OLMo-7B-0424-hf')

In [14]:
max_new_tokens = 10

In [28]:
for i in range(kwargs["topk"]):
    pos = indices[i,1].item()
    results = []
    results.append("Input tokens:")
    results.append(dumps([tokenizer.decode(t) for t in my_input_ids[i][:pos+1]]))
    results.append("Ground-truth output token:")
    results.append(tokenizer.decode(my_input_ids[i][pos+1]))
    if max_new_tokens>1:
        results.append("Ground-truth continuation:")
        results.append(tokenizer.decode(my_input_ids[i][pos+1:pos+1+max_new_tokens]))
    print('\n'.join(results))

Input tokens:
["<|endoftext|>", "Lu", "is", " Su", "arez", "\u2019", "s", " European", " career", " will", " come", " to", " an", " end", " after", " reaching", " an", " agreement", " with", " his", " boy", "hood", " club", " Nacional", ".", " The", " player", " spent", " 15", " years", " in", " Europe", ",", " playing", " for", " G", "ron", "ingen", ",", " Ajax"]
Ground-truth output token:
,
Ground-truth continuation:
, Liverpool, Barcelona, and Atletico Madrid
Input tokens:
["<|endoftext|>", "Would", " like", " some", " insight", " or"]
Ground-truth output token:
 tips
Ground-truth continuation:
 tips with starting my non-advertiser friendly
Input tokens:
["<|endoftext|>", "Log", "o", " of", " the", " United", " States", " Agency", " for", " International", " Development", " (", "USA", "ID", ").", "\n", "Direct", "s", " USA", "ID", " to", " report", " annually", " on", " the", " use", " of", " global", " health", " innovations", " in", " its"]
Ground-truth output token:
 programs
Gro

## Making it into a function

In [34]:
def find_and_show_texts(topk=4, **kwargs):
    indices = find_texts(
        metric="loss",
        data_dir=DATA_DIR,
        extremum="min",
        topk=topk,
        **kwargs,
    )
    my_input_ids = text_dataset['input_ids'][indices[:,0]]
    results = []
    for i in range(topk):
        pos = indices[i,1].item()
        results.append("Input tokens:")
        results.append(tokenizer.decode(my_input_ids[i][:pos+1]))
        results.append("Ground-truth output token:")
        results.append(tokenizer.decode(my_input_ids[i][pos+1]))
        if max_new_tokens>1:
            results.append("Ground-truth continuation:")
            results.append(tokenizer.decode(my_input_ids[i][pos+1:pos+1+max_new_tokens]))
    return results


In [35]:
for neuron_subset_name in ["weakening", "conditional strengthening", "proportional change", "conditional weakening", "orthogonal output", "strengthening"]:
    for intervention_type in ["mean_ablation", "zero_ablation"]:
        results = find_and_show_texts(neuron_subset_name=neuron_subset_name, intervention_type=intervention_type)
        print('===================================================')
        print(neuron_subset_name.upper(), intervention_type.upper())
        print('\n'.join(results))

WEAKENING MEAN_ABLATION
Input tokens:
<|endoftext|>Luis Suarez’s European career will come to an end after reaching an agreement with his boyhood club Nacional. The player spent 15 years in Europe, playing for Groningen, Ajax
Ground-truth output token:
,
Ground-truth continuation:
, Liverpool, Barcelona, and Atletico Madrid
Input tokens:
<|endoftext|>Would like some insight or
Ground-truth output token:
 tips
Ground-truth continuation:
 tips with starting my non-advertiser friendly
Input tokens:
<|endoftext|>Logo of the United States Agency for International Development (USAID).
Directs USAID to report annually on the use of global health innovations in its
Ground-truth output token:
 programs
Ground-truth continuation:
 programs.
This bill directs the Administrator of the
Input tokens:
<|endoftext|>
Ground-truth output token:
Check
Ground-truth continuation:
Check the temperature of the water that is dripping from
WEAKENING ZERO_ABLATION
Input tokens:
<|endoftext|>The people who 
live